<a href="https://colab.research.google.com/github/hanamonem/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hanamonem/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

## Unit of Analysis

One row represents one content item (content_hash_id) for one reporting date (report_date).

Time window:
March 2026 (month=2026-03).

Prediction target:
Future organic performance measured through GSC clicks.

Excluded:
June 2026 data is excluded because it is reserved as a holdout/test month.

In [15]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
TYPE huggingface,
TOKEN '{HF_TOKEN}'
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet(
'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 2. Fields: feature / label / context / excluded

### Features
- gsc_impressions
- gsc_avg_position
- ga4_pageviews
- ga4_sessions
- ga4_users

### Label
- future_clicks (future performance proxy)

### Context
- report_date
- client_hash_id
- content_hash_id
- month

### Excluded
- gsc_clicks_future (would leak target information)
- month=2026-06 data (reserved holdout month)

In [16]:
sample = con.sql(f"""
SELECT *
FROM read_parquet(
'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'
)
LIMIT 5
""").df()

print(sample.columns.tolist())

sample.head()

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## 3. Verify it with queries (grain, counts, missing values, windows)

Query 1 verifies the grain.

Query 2 verifies row count and date span.

Query 3 verifies availability using IS TRUE.

In [23]:
con.sql(f"""
SELECT
COUNT(*) AS rows,
COUNT(DISTINCT content_hash_id) AS unique_content
FROM read_parquet(
'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,unique_content
0,9841378,331437


In [24]:
con.sql(f"""
SELECT
COUNT(*) AS rows,
MIN(report_date) AS start_date,
MAX(report_date) AS end_date
FROM read_parquet(
'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31


In [25]:
con.sql(f"""
SELECT
COUNT(*) AS available_rows
FROM read_parquet(
'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE gsc_data_available IS TRUE
""").df()

,available_rows
0,3611061


# Five Features
1. gsc_impressions
   Available at decision time because impressions are observed before prediction.

2. gsc_avg_position
   Available at decision time because historical ranking data already exists.

3. ga4_pageviews
   Available at decision time because pageviews are collected before prediction.

4. ga4_sessions
   Available at decision time because sessions are historical observations.

5. ga4_users
   Available at decision time because users are measured before the prediction date.

In [26]:
features = con.sql(f"""
SELECT
gsc_impressions,
gsc_avg_position,
ga4_pageviews,
ga4_sessions,
ga4_users
FROM read_parquet(
'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE gsc_data_available IS TRUE
LIMIT 1000
""").df()

features.head()

,gsc_impressions,gsc_avg_position,ga4_pageviews,ga4_sessions,ga4_users
0,20,3.350000,<NA>,<NA>,<NA>
1,1,0.000000,<NA>,<NA>,<NA>
2,125,4.928000,<NA>,<NA>,<NA>
3,7,4.000000,<NA>,<NA>,<NA>
4,11,2.272727,<NA>,<NA>,<NA>


# Leakage Trap
Leakage experiment:

I deliberately added the target information as a feature.
The evaluation score increased sharply because the model could directly access future information.

This is target leakage and would not be available at prediction time.

The leaked feature was removed and only the honest score was retained.

In [27]:
df = con.sql(f"""
SELECT
gsc_impressions,
gsc_avg_position,
ga4_pageviews,
gsc_clicks
FROM read_parquet(
'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE gsc_data_available IS TRUE
LIMIT 1000
""").df()

# honest features
X = df[['gsc_impressions','gsc_avg_position','ga4_pageviews']]

# target
y = df['gsc_clicks']

# leaked feature
df['leak_feature'] = y

## 4. Data limits
Limitations:

1. The dataset is an unbalanced panel and clients have different history lengths.

2. Some clients only have GSC access while others have both GSC and GA4.

3. This analysis uses only March 2026 and may not capture seasonal effects.

4. Observational data can support decisions but cannot prove causality.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [21]:
con.sql(f"""
SELECT *
FROM read_parquet(
'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'
)
LIMIT 1
""").df().columns.tolist()

['report_date',
 'client_hash_id',
 'content_hash_id',
 'client_has_gsc',
 'client_has_ga4',
 'gsc_data_available',
 'ga4_data_available',
 'gsc_impressions',
 'gsc_clicks',
 'gsc_sum_position',
 'gsc_avg_position',
 'ga4_pageviews',
 'ga4_sessions',
 'ga4_users',
 'ga4_engaged_sessions',
 'ga4_total_engagement_sec',
 'sessions_organic',
 'sessions_direct',
 'sessions_referral',
 'sessions_social',
 'sessions_paid',
 'sessions_ai',
 'ai_chatgpt',
 'ai_perplexity',
 'ai_gemini',
 'ai_copilot',
 'ai_claude',
 'ai_meta',
 'ai_other',
 'scroll_events',
 'month']

In [22]:
for c in sample.columns:
    print(c)


report_date
client_hash_id
content_hash_id
client_has_gsc
client_has_ga4
gsc_data_available
ga4_data_available
gsc_impressions
gsc_clicks
gsc_sum_position
gsc_avg_position
ga4_pageviews
ga4_sessions
ga4_users
ga4_engaged_sessions
ga4_total_engagement_sec
sessions_organic
sessions_direct
sessions_referral
sessions_social
sessions_paid
sessions_ai
ai_chatgpt
ai_perplexity
ai_gemini
ai_copilot
ai_claude
ai_meta
ai_other
scroll_events
month
